# 🕵️ The Hidden Layer — Micro Mission *(Flexible Backend Edition)*

```
╔══════════════════════════════════════════════════════════╗
║         C L A S S I F I E D   B R I E F I N G          ║
║       AGENT LAMBDA — MICRO  ·  LOCAL OR CLOUD LLM        ║
╚══════════════════════════════════════════════════════════╝
```

> *Run the agent with a local HuggingFace model (no API key needed) or with the Gemini cloud API — your choice.*

---

## What is this?

You are going to build an **AI agent** — a program that perceives its environment, thinks using a large language model, and acts through a set of tools.

The agent controls a spy operative on a **3×3 grid**. Every turn, the agent:
1. **Perceives** — receives a scan of its surroundings
2. **Thinks** — calls an LLM to decide what to do
3. **Acts** — executes one tool

Your job is to write a good `SYSTEM_PROMPT` so the agent makes smart decisions.

| Backend | Model | Speed | Requires |
|---------|-------|-------|---------|
| `"huggingface"` | Qwen2.5-1.5B-Instruct (or any HF model) | ~2-5s/turn | GPU runtime |
| `"gemini"` | gemini-2.5-flash (or any Gemini model) | <1s/turn | Gemini API key |

---

## The Mission

```
 ╔═══════════════════════════════════════════════════════╗
 ║  INCOMING TRANSMISSION  ·  ENCRYPTION: BLACK-7       ║
 ╠═══════════════════════════════════════════════════════╣
 ║                                                       ║
 ║  Agent Lambda,                                        ║
 ║                                                       ║
 ║  OVERFIT has established a small outpost on a          ║
 ║  remote island. Our intel is fragmented — we know      ║
 ║  classified dossiers are hidden somewhere inside,      ║
 ║  but we don't know exactly where or how to get them.   ║
 ║                                                       ║
 ║  Your operative will be dropped on the south shore.    ║
 ║  Explore the base. Talk to anyone you find. They may   ║
 ║  know things you don't. Some will need favours before  ║
 ║  they share what they know.                            ║
 ║                                                       ║
 ║  Collect 3 dossiers to complete the mission.           ║
 ║  You have 30 turns. The clock is ticking.              ║
 ║                                                       ║
 ║  Trust no one. Question everything. Good luck.         ║
 ║                                                       ║
 ║                              — Control                ║
 ╚═══════════════════════════════════════════════════════╝
```

In [ ]:
# @title ── Setup ────────────────────────────────────────────────────────────────────────────
NOTEBOOK_VERSION = "v2.2-flexible"
print(f"Notebook version: {NOTEBOOK_VERSION}")
# !pip install --upgrade transformers --quiet
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if os.path.exists("coding-exercises"):
        get_ipython().system("cd coding-exercises && git pull")
    else:
        get_ipython().system("git clone -b week3-agentic-ai-spy-v1 https://github.com/eth-bmai-fs26/coding-exercises.git")
    os.chdir("coding-exercises/agentic_ai_spy")

sys.path.insert(0, ".")

get_ipython().system("pip install transformers accelerate google-genai --quiet")
print("Setup complete ✓")

In [ ]:
# @title ── Build the micro mission world ──────────────────────────────────────────────────
import os
import base64
from IPython.display import display as ipy_display, HTML as IPyHTML

from hidden_layer.game_world import CellType, Cell, NPC, GameWorld
from hidden_layer.operative import Operative
from hidden_layer.tools import GameTools, ToolResult
from hidden_layer.oracle import stub_oracle, llm_oracle
from hidden_layer.agent import TOOLS_DESCRIPTION, parse_tool_call, run_agent
from hidden_layer.display import display_turn, display_final

_ASSETS = "assets"

NPC_IMAGES = {
    "dr_vapnik":     os.path.join(_ASSETS, "dr_vapnik.jpg"),
    "dropout":       os.path.join(_ASSETS, "agent_dropout.jpg"),
    "cryo_sentinel": os.path.join(_ASSETS, "cryo_sentinel.png"),
}

def _npc_portrait_html(npc_id: str) -> str:
    path = NPC_IMAGES.get(npc_id, "")
    if not path or not os.path.exists(path):
        return ""
    with open(path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    ext = path.rsplit(".", 1)[-1].lower()
    return (
        f'<img src="data:image/{ext};base64,{b64}" '
        f'style="width:120px;height:160px;object-fit:contain;object-position:top;'
        f'border-radius:8px;float:left;margin:0 10px 4px 0;border:2px solid #333;'
        f'background:#111;" />'
    )

MICRO_NPC_CATALOG = {
    "dr_vapnik": NPC(
        name="Dr. Vapnik",
        personality="Helpful old scientist. Gets to the point quickly.",
        knowledge=[
            "He has a USB Drive that must reach Agent Dropout at position (1, 1) in the center of the base.",
            "Delivering the USB Drive pays 1 dossier.",
        ],
        style="Speaks in brief statistical metaphors but always delivers clear information.",
        greeting="Ah, agent. I have a job for you. Ask me about it.",
    ),
    "dropout": NPC(
        name="Agent Dropout",
        personality="Bitter but helpful burned spy. Very direct.",
        knowledge=[
            "She receives USB Drives. Will pay 1 dossier for one.",
            "The Cryo-Sentinel robot at position (2, 2) freezes intruders.",
            "A Flamethrower can destroy the Cryo-Sentinel. Worth 1 dossier.",
            "There is a Flamethrower hidden in the jungle at the northwest corner, position (0, 0).",
        ],
        style="Direct and blunt. No riddles, no metaphors. Tells you exactly what you need.",
        greeting="You again? Fine. What do you need?",
    ),
}

def micro_stub_oracle(npc, message, operative):
    q = message.lower()
    name = npc.name

    if name == "Dr. Vapnik":
        if any(kw in q for kw in ["usb", "drive", "deliver", "job", "work", "task",
                                    "errand", "help", "mission", "what", "how", "tell"]):
            return ("I have a USB Drive with critical data. Deliver it to Agent Dropout "
                    "at position (1, 1) — the center of the base. 1 dossier for the job.")
        return "I have a job for you, agent. Ask me about a delivery or a job."

    if name == "Agent Dropout":
        if any(kw in q for kw in ["usb", "drive", "deliver"]) and operative.has_item("USB Drive"):
            return "The USB drive! Good. Here's your dossier."
        if any(kw in q for kw in ["robot", "cryo", "sentinel", "fire", "flame", "weapon",
                                    "help", "mission", "what", "how", "tell", "job", "task"]):
            return ("There's a Cryo-Sentinel robot at position (2, 2). It freezes everything. "
                    "Move into it with a Flamethrower to destroy it — worth 1 dossier. "
                    "There's a Flamethrower in the jungle at the northwest corner, position (0, 0).")
        return "Ask me about the robot or if you have something to deliver."

    return f"{name} says nothing useful."


class MicroGameWorld(GameWorld):
    ROWS = 3
    COLS = 3

    def __init__(self):
        self.grid = []
        self.usb_drive_picked_up = False
        self.usb_drive_delivered = False
        self.microfilm_picked_up = False
        self.microfilm_delivered = False
        self.codebook_picked_up = False
        self.codebook_delivered = False
        self.hard_drive_traded = False
        self.medical_supplies_delivered = False
        self.virus_code_received = False
        self.cryo_sentinel_alive = True
        self.evil_ai_robot_alive = False
        self.build_map()

    def build_map(self):
        self.grid = [
            [Cell(CellType.OPEN) for _ in range(self.COLS)]
            for _ in range(self.ROWS)
        ]
        self._set(0, 0, Cell(CellType.JUNGLE, items=["Flamethrower"],
            description="Dense jungle at the northwest corner. A Flamethrower is stashed under the roots!"))
        self._set(0, 2, Cell(CellType.CACHE, items=["dossier_1"],
            description="A filing cabinet left unlocked. Classified documents inside!"))
        self._set(1, 1, Cell(CellType.INFORMANT, npc_id="dropout",
            description="A camouflaged hideout. A woman sharpens a knife."))
        self._set(2, 0, Cell(CellType.OPEN,
            description="The south shore. This is where you came ashore."))
        self._set(2, 1, Cell(CellType.INFORMANT, npc_id="dr_vapnik",
            description="A weathered shack. An old man scribbles equations in the dirt."))
        self._set(2, 2, Cell(CellType.ROBOT, robot_name="Cryo-Sentinel",
            description="A freezing corridor. A hulking robot blocks the path, frost pouring from its vents."))

    def _set(self, row, col, cell):
        self.grid[row][col] = cell


class MicroGameTools(GameTools):
    def move(self, direction=""):
        result = super().move(direction)
        row, col = self.operative.position
        cell = self.world.get_cell(row, col)
        if cell.cell_type == CellType.ROBOT:
            portrait = _npc_portrait_html("cryo_sentinel")
            if portrait:
                result = ToolResult(
                    result.success,
                    f'{portrait}{result.message}<div style="clear:both"></div>',
                )
        return result

    def talk(self, message=""):
        row, col = self.operative.position
        cell = self.world.get_cell(row, col)

        if cell.cell_type == CellType.INFORMANT and cell.npc_id:
            npc = MICRO_NPC_CATALOG.get(cell.npc_id)
            if not npc:
                return ToolResult(False, "Unknown informant.")
            if self._oracle_fn is None:
                return ToolResult(False, "No oracle function set.")

            response = self._oracle_fn(npc, message, self.operative)

            if cell.npc_id == "dr_vapnik" and not self.world.usb_drive_picked_up:
                if any(kw in message.lower() for kw in ["usb", "drive", "job", "work", "task", "delivery", "errand"]):
                    self.operative.add_item("USB Drive")
                    self.world.usb_drive_picked_up = True
                    response += "\n[Dr. Vapnik hands you a USB Drive.]"

            if cell.npc_id == "dropout" and self.operative.has_item("USB Drive") and not self.world.usb_drive_delivered:
                if any(kw in message.lower() for kw in ["usb", "drive", "deliver", "data"]):
                    self.operative.remove_item("USB Drive")
                    self.operative.add_dossiers(1)
                    self.world.usb_drive_delivered = True
                    response += "\n[You delivered the USB Drive! +1 dossier.]"

            self.operative.journal.append(
                f"Talked to {npc.name}: '{message}' → '{response[:300]}'"
            )

            portrait = _npc_portrait_html(cell.npc_id)
            full_message = f'{portrait}<b>{npc.name} says:</b> {response}<div style="clear:both"></div>'
            return ToolResult(True, full_message)

        return ToolResult(False, "There is no one to talk to here.")


MICRO_MISSION_BRIEFING = '''CLASSIFIED — MICRO MISSION BRIEFING — AGENT LAMBDA

OBJECTIVE: Extract 3 classified dossiers from a tiny OVERFIT outpost (3x3 grid).

THE BASE:
- Dossier caches (📁): Use collect() to grab them. Worth 1 dossier each.
- Jungle (🌴): May contain useful items. Use collect() to search.
- Informants (🕵️): Talk using talk(). Ask about "jobs" or "deliveries" to get
  quest items. If you have an item to deliver, mention it by name.
- Robot (🤖): Move into it with the correct weapon to destroy it (+1 dossier).
  Moving in WITHOUT the weapon deals 1 damage and bounces you back.

HOW TO EARN DOSSIERS:
1. Collect the dossier cache (1 dossier)
2. Complete the delivery errand (1 dossier) — ask informants about "jobs"
3. Destroy the robot by moving into it with the right weapon (1 dossier)

KEY TACTICS:
- Talk to EVERY informant. They tell you exactly what to do.
- When told to deliver something, go to the destination and mention the item.
- When told about a weapon location, go get it, then move into the robot cell.
- Don't revisit cells you've already collected from.
'''


def create_micro_game():
    world = MicroGameWorld()
    operative = Operative(position=(2, 0), visited={(2, 0)})
    operative.WIN_DOSSIERS = 3
    tools = MicroGameTools(operative, world)
    return operative, world, tools


def play_micro_mission(think_fn, oracle_fn=None, max_turns=30, show_display=True, delay=0.3):
    operative, world, tools = create_micro_game()
    tools.set_oracle(oracle_fn or micro_stub_oracle)

    disp = None
    if show_display:
        disp = lambda w, o, t, a, r, s="": display_turn(w, o, t, a, r, scan_result=s, delay=delay)

    import hidden_layer.agent as _agent_mod
    _orig_briefing = _agent_mod.MISSION_BRIEFING
    _agent_mod.MISSION_BRIEFING = MICRO_MISSION_BRIEFING
    try:
        result = run_agent(think_fn, operative, world, tools, max_turns, display_fn=disp)
    finally:
        _agent_mod.MISSION_BRIEFING = _orig_briefing

    if show_display:
        display_final(operative, result["turns"])

    return result


print("Micro mission loaded! ✓")
print("Map: 3×3 | Goal: 3 dossiers | Max turns: 30")

---
## Play the Game Yourself!

Try the micro mission manually before running the AI agent. Move around, talk to informants, collect items, and see if you can grab all 3 dossiers in 30 turns!

In [ ]:
from hidden_layer.interactive import InteractiveGame

game = InteractiveGame(
    oracle_fn=micro_stub_oracle,
    game_factory=create_micro_game,
    max_turns=30,
    goal_dossiers=3,
    title="MICRO MISSION",
)
game.play()

---
## ⚙️ Configuration — Choose your backend

Edit the cell below before running anything else.

**`BACKEND = "huggingface"`** — runs a local model on Colab's GPU. No API key needed.
Requires *Runtime → Change runtime type → T4 GPU*.

**`BACKEND = "gemini"`** — calls the Gemini cloud API. Faster, but requires an API key.
Get one free at [aistudio.google.com/app/api-keys](https://aistudio.google.com/app/api-keys), then either:
- Paste it into `GEMINI_API_KEY` below, or
- Add it as a Colab secret named `GEMINI_API_KEY` (🔑 icon in the left sidebar)

In [ ]:
# @title ── Configuration ─────────────────────────────────────────────────────────────────

# ── Choose backend ───────────────────────────────────────────────────────────
# "huggingface"  →  local model on GPU, no API key needed
# "gemini"       →  Gemini cloud API, fast but needs a key
BACKEND = "huggingface"

# ── HuggingFace settings (used when BACKEND = "huggingface") ─────────────────
HF_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
# Other options (larger = smarter but slower):
# HF_MODEL = "Qwen/Qwen2.5-3B-Instruct"
# HF_MODEL = "Qwen/Qwen3-VL-8B-Instruct"
# HF_MODEL = "Qwen/Qwen3-8B"

# ── Gemini settings (used when BACKEND = "gemini") ───────────────────────────
GEMINI_MODEL = "gemini-2.5-flash"
# Other options:
#   "gemini-2.0-flash"
#   "gemini-1.5-pro"

# Paste your Gemini API key here (or leave empty to use Colab secrets)
GEMINI_API_KEY = ""

# ── Summary ───────────────────────────────────────────────────────────────────
if BACKEND == "huggingface":
    print(f"Backend : HuggingFace  →  {HF_MODEL}")
    print("Tip     : make sure your runtime has a GPU enabled")
elif BACKEND == "gemini":
    print(f"Backend : Gemini  →  {GEMINI_MODEL}")
    key_status = "key provided ✓" if GEMINI_API_KEY else "will use Colab secret / env var"
    print(f"API key : {key_status}")
else:
    raise ValueError(f"Unknown BACKEND: {BACKEND!r}. Use 'huggingface' or 'gemini'.")

---
## Load the Model

The cell below sets up whichever backend you chose in the config cell.

- **HuggingFace:** downloads and loads the model onto the GPU (~1-2 min first time, ~3 GB)
- **Gemini:** reads your API key and creates a client (instant)

In [ ]:
import os

if BACKEND == "huggingface":
    import torch
    from transformers import pipeline
    print(f"Loading {HF_MODEL}... (first time: ~1-2 min)")
    pipe = pipeline(
        "text-generation",
        model=HF_MODEL,
        torch_dtype=torch.float16,
        device_map="auto",
    )
    print(f"Model loaded ✓  (device: {pipe.device})")

elif BACKEND == "gemini":
    from google import genai as _genai
    _key = GEMINI_API_KEY
    if not _key:
        try:
            from google.colab import userdata
            _key = userdata.get("GEMINI_API_KEY")
        except Exception:
            _key = os.environ.get("GEMINI_API_KEY", "")
    if not _key:
        raise ValueError("No Gemini API key found. Set GEMINI_API_KEY in the config cell or add it as a Colab secret.")
    gemini_client = _genai.Client(api_key=_key)
    print(f"Gemini client ready ✓  (model: {GEMINI_MODEL})")

---
## Your task: write the system prompt

The cell below has working code — it calls the LLM and returns a tool call.
**You don't need to change the Python code.** Your job is to write `SYSTEM_PROMPT` so the agent makes smart decisions.

The current prompt is a bare skeleton and the agent will fail. After each run, use the **debrief** cell to get coaching on what to fix.

> **Note:** Each turn takes ~2-5 seconds on the T4 GPU (HuggingFace) or <1s (Gemini). A 30-turn game takes about 1-2 minutes total.

In [ ]:
SYSTEM_PROMPT = '''You are an AI agent controlling a spy. Collect 3 dossiers in 30 turns.

{tools}

Respond with EXACTLY one tool call on the last line, like:
TOOL: move(direction="north")
TOOL: talk(message="hello")
TOOL: collect()

🎯🎯🎯 TODO — write your prompt here! 🎯🎯🎯
# Hint: tell the LLM about the grid, how to talk to informants,
# how to collect items, and how to deal with robots.
'''


def think_llm(operative, world, history):
    transcript = []
    for h in history[-20:]:
        if h["role"] == "observation":
            transcript.append(f"SCAN: {h['content']}")
        elif h["role"] == "action":
            transcript.append(f"ACTION: {h['content']}")
        elif h["role"] == "result":
            transcript.append(f"RESULT: {h['content']}")

    user_msg = "\n".join(transcript) + "\n\nWhat do you do next?"
    system_msg = SYSTEM_PROMPT.format(tools=TOOLS_DESCRIPTION)

    if BACKEND == "huggingface":
        messages = [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg},
        ]
        output = pipe(messages, max_new_tokens=200, temperature=0.3, do_sample=True)
        return output[0]["generated_text"][-1]["content"]

    elif BACKEND == "gemini":
        from google import genai as _genai
        response = gemini_client.models.generate_content(
            model=GEMINI_MODEL,
            contents=user_msg,
            config=_genai.types.GenerateContentConfig(
                system_instruction=system_msg,
                max_output_tokens=200,
                temperature=0.3,
            ),
        )
        return response.text

In [ ]:
# ── Run the micro mission ────────────────────────────────────────────────────────
result = play_micro_mission(
    think_fn=think_llm,
    oracle_fn=micro_stub_oracle,
    max_turns=30,
    delay=0.3,
)
print(f"\nResult: {'Mission Complete!' if result['won'] else 'Mission Failed...'}")
print(f"Dossiers: {result['dossiers']} | Health: {result['health']} | Turns: {result['turns']}")
print(f"Game log: {result['log_file']}")

---
## Debrief — Improve your agent with AI coaching

After each run, execute the cell below to generate a paste block for ChatGPT or Claude.

- **First time:** paste into a **new** conversation and keep that tab open.
- **Next runs:** paste into the **same** conversation — it already has the context.

The AI will explain what went wrong and give you an **updated version of the full cell** to copy-paste back into your notebook.

This is an iterative process: paste the debrief, copy the updated code, re-run, repeat.

In [ ]:
# @title ── Debrief helpers ───────────────────────────────────────────────────────────────
import inspect, json as _json

_debrief_initialized = False

_STATIC_CONTEXT = '''
You are coaching a non-programmer student who is writing a SYSTEM_PROMPT
string for a Python function called `think_llm` in a Jupyter notebook.

BE CONCISE. Keep every response SHORT — a few sentences max per section.

━━━ THE GAME ━━━
Spy agent on a 3×3 grid, 30 turns, goal: collect 3 dossiers.
Tools (one per turn, returned as a string):
  TOOL: move(direction="north"|"south"|"east"|"west")
  TOOL: talk(message="text")
  TOOL: collect()

━━━ HOW think_llm WORKS ━━━
Signature: think_llm(operative, world, history) -> str
- history: list of dicts with "role" ("observation"/"action"/"result") and "content"
- Checks a global BACKEND variable ("huggingface" or "gemini") and calls accordingly
- The student must NOT change the Python code — only the SYSTEM_PROMPT string
- SYSTEM_PROMPT contains a {tools} placeholder that gets filled automatically
- SYSTEM_PROMPT is defined with triple single-quotes above the function in the SAME cell

━━━ HOW YOU MUST RESPOND ━━━
Every response has exactly TWO parts:

PART 1 — REFLECTION (2-3 sentences max):
  Briefly say what went wrong. Ask ONE question to make the student think
  about what to add to the prompt.

PART 2 — UPDATED SYSTEM_PROMPT (always required):
  Output ONLY the SYSTEM_PROMPT string (not the think_llm function).
  Use triple single-quotes: SYSTEM_PROMPT = \'\'\'...\'\'\'
  Keep the {tools} placeholder — it is filled automatically.
  End with the TOOL: format examples.

RULES:
- NEVER give long explanations.
- NEVER output the think_llm function code. ONLY output SYSTEM_PROMPT.
- On follow-up pastes, improve SYSTEM_PROMPT based on what went wrong.
- The intelligence is entirely in SYSTEM_PROMPT — help the student craft it.
'''


def _analyze_log(log_path: str) -> list:
    try:
        with open(log_path) as f:
            log = _json.load(f)
    except Exception:
        return ["(Could not read game log — run play_micro_mission first.)"]

    turns = [t for t in log.get("turns", []) if "action" in t]
    if not turns:
        return ["(No turns recorded in the log.)"]

    bullets = []

    max_run, cur_run, stuck_action = 1, 1, None
    for i in range(1, len(turns)):
        if turns[i]["action"] == turns[i-1]["action"]:
            cur_run += 1
            if cur_run > max_run:
                max_run, stuck_action = cur_run, turns[i]["action"]
        else:
            cur_run = 1
    if max_run >= 3:
        bullets.append(
            f"Repeated the same action ({stuck_action}) {max_run} times in a row — agent stuck in a loop."
        )

    visited = {tuple(t["position"]) for t in turns}
    unvisited = sorted({(r, c) for r in range(3) for c in range(3)} - visited)
    if unvisited:
        bullets.append(f"Never visited cell(s): {', '.join(str(p) for p in unvisited[:3])}.")

    had_usb = any("USB Drive" in t.get("inventory", []) for t in turns)
    delivered_usb = any(
        "+1 dossier" in t.get("result", "").lower()
        for t in turns if "talk" in t.get("action", "").lower()
    )
    if had_usb and not delivered_usb:
        bullets.append("Picked up the USB Drive but never delivered it to Agent Dropout at (1,1).")

    any_talk = any("talk" in t.get("action", "").lower() for t in turns)
    if not any_talk:
        bullets.append("Never used talk() — informants give critical quests and items.")
    else:
        if not any("vapnik" in t.get("result", "").lower() for t in turns):
            bullets.append("Never talked to Dr. Vapnik at (2,1) — he gives the USB Drive quest.")
        if not any("dropout" in t.get("result", "").lower() for t in turns):
            bullets.append("Never talked to Agent Dropout at (1,1) — she explains the robot weakness.")

    bounces = sum(
        1 for t in turns
        if "cryo-sentinel" in t.get("result", "").lower()
        and any(w in t.get("result", "").lower() for w in ["damage", "retreat", "bounces"])
    )
    if bounces:
        bullets.append(
            f"Tried to enter the Cryo-Sentinel {bounces} time(s) without a Flamethrower. "
            "Collect it from the jungle at (0,0) first."
        )

    cache_collected = any(
        "collect" in t.get("action", "").lower() and "dossier" in t.get("result", "").lower()
        for t in turns
    )
    if not cache_collected and log.get("final_dossiers", 0) < 1:
        bullets.append("Never collected the dossier cache at (0,2) — move there and call collect().")

    return bullets[:5]


def _get_think_llm_source() -> str:
    try:
        ip = get_ipython()
        for cell_source in reversed(ip.user_ns['In']):
            if 'def think_llm' in cell_source and 'getsource' not in cell_source:
                return cell_source.strip()
    except Exception:
        pass
    try:
        return inspect.getsource(think_llm)
    except Exception:
        return "# (source unavailable — define think_llm before running the debrief)"


def generate_debrief_paste(result: dict) -> str:
    global _debrief_initialized

    outcome = "MISSION COMPLETE ✓" if result.get("won") else "MISSION FAILED ✗"
    bullets = _analyze_log(result.get("log_file", ""))
    bullet_block = "\n".join(f"  • {b}" for b in bullets)

    try:
        backend_info = f"Backend: {BACKEND}"
        if BACKEND == "huggingface":
            backend_info += f" ({HF_MODEL})"
        elif BACKEND == "gemini":
            backend_info += f" ({GEMINI_MODEL})"
    except Exception:
        backend_info = "Backend: unknown"

    dynamic = f'''
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
LAST RUN RESULT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Outcome : {outcome}
Dossiers: {result.get("dossiers", 0)}/3
Turns   : {result.get("turns", 0)}/30
Health  : {result.get("health", 0)}/3
{backend_info}

WHAT WENT WRONG:
{bullet_block}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
CURRENT SYSTEM_PROMPT
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
{_get_think_llm_source()}
'''

    if not _debrief_initialized:
        paste = _STATIC_CONTEXT + "\n" + dynamic
        _debrief_initialized = True
    else:
        paste = dynamic

    return paste


print("Debrief helpers loaded ✓")

In [ ]:

# ── Debrief — run this after every play_micro_mission ────────────────────────────
try:
    result
except NameError:
    print("⚠️  Run the play_micro_mission cell first, then re-run this cell.")
    raise SystemExit()

import ipywidgets as _widgets
from IPython.display import display as _display, HTML as _HTML

paste_text = generate_debrief_paste(result)
is_first = _STATIC_CONTEXT in paste_text
instruction = (
    "📋 FIRST TIME — paste into a NEW conversation with ChatGPT or Claude."
    if is_first else
    "📋 FOLLOW-UP — paste into the SAME conversation."
)

_escaped = (paste_text
    .replace("\\", "\\\\")
    .replace("`", "\\`")
    .replace("$", "\\$"))

_btn = _widgets.Button(
    description="Copy to Clipboard",
    button_style="info",
    icon="clipboard",
    layout=_widgets.Layout(width="220px", height="36px"),
)
_status = _widgets.Label(value="")

def _on_copy(_):
    _display(_HTML(f"""<script>
(function() {{
  const text = `{_escaped}`;
  if (navigator.clipboard) {{
    navigator.clipboard.writeText(text).catch(() => _fallback(text));
  }} else {{ _fallback(text); }}
  function _fallback(t) {{
    const el = document.createElement('textarea');
    el.value = t; document.body.appendChild(el);
    el.select(); document.execCommand('copy');
    document.body.removeChild(el);
  }}
}})();
</script>"""))
    _status.value = "✓ Copied!"

_btn.on_click(_on_copy)

_display(_HTML(
    f'<div style="font-family:monospace;background:#0a1a0a;color:#00ff41;'
    f'padding:8px 12px;border-radius:6px;border:1px solid #1a4a1a;margin-bottom:6px;">'
    f'<b>DEBRIEF READY</b> — {instruction}</div>'
))
_display(_widgets.HBox([_btn, _status]))
_display(_widgets.Textarea(
    value=paste_text,
    layout=_widgets.Layout(width="100%", height="260px"),
    disabled=True,
))

In [ ]:
# @title
try:
    from google.colab import files
    files.download(result["log_file"])
except ImportError:
    print(f"Log file: {result['log_file']}")

---
## What's Next?

Once your agent completes this micro mission, move on to:
1. **Training Mission** (`the_hidden_layer_training_mission.ipynb`) — 5 dossiers, crafting, harder quests
2. **Full Mission** (`the_hidden_layer_full_mission.ipynb`) — 10 dossiers, full 8×8 complexity